In [0]:
# Core Spark & utility imports
from pyspark.sql.functions import col, when
from datetime import datetime
import uuid

In [0]:
# Create a widget to simulate pipeline parameter
dbutils.widgets.text("Environment", "silver")

# Retrieve parameter value
environment = dbutils.widgets.get("Environment")

In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import col, when
from datetime import datetime

# Current timestamp (simulating InsertedDate / ModifiedDate)
now_date = datetime.now()

# Data equivalent to config.tbl_Entities inserts
data = [
    (1,  'Sales.CountryRegionCurrency','silver','fabmigration','fabmigration','bronze/adventureworks2','parquet','silver/adventureworks','01_SilverCleanEntity'),
    (2,  'Sales.CreditCard','silver','fabmigration','fabmigration','bronze/adventureworks','parquet','silver/adventureworks','01_SilverCleanEntity'),
    (3,  'Sales.Currency','silver','fabmigration','fabmigration','bronze/adventureworks2','parquet','silver/adventureworks','01_SilverCleanEntity'),
    (4,  'Sales.CurrencyRate','silver','fabmigration','fabmigration','bronze/adventureworks2','parquet','silver/adventureworks','01_SilverCleanEntity'),
    (5,  'Sales.Customer','silver','fabmigration','fabmigration','bronze/adventureworks2','parquet','silver/adventureworks','01_SilverCleanEntity'),
    (6,  'Sales.PersonCreditCard','silver','fabmigration','fabmigration','bronze/adventureworks2','parquet','silver/adventureworks','01_SilverCleanEntity'),
    (7,  'Sales.SalesOrderDetail','silver','fabmigration','fabmigration','bronze/adventureworks2','parquet','silver/adventureworks','01_SilverCleanEntity'),
    (8,  'Sales.SalesOrderHeader','silver','fabmigration','fabmigration','bronze/adventureworks2','parquet','silver/adventureworks','01_SilverCleanEntity'),
    (9,  'Sales.SalesOrderHeaderSalesReason','silver','fabmigration','fabmigration','bronze/adventureworks2','parquet','silver/adventureworks','01_SilverCleanEntity'),
    (10, 'Sales.SalesPerson','silver','fabmigration','fabmigration','bronze/adventureworks2','parquet','silver/adventureworks','01_SilverCleanEntity'),
    (11, 'Sales.SalesPersonQuotaHistory','silver','fabmigration','fabmigration','bronze/adventureworks2','parquet','silver/adventureworks','01_SilverCleanEntity'),
    (12, 'Sales.SalesReason','silver','fabmigration','fabmigration','bronze/adventureworks2','parquet','silver/adventureworks','01_SilverCleanEntity'),
    (13, 'Sales.SalesTaxRate','silver','fabmigration','fabmigration','bronze/adventureworks2','parquet','silver/adventureworks','01_SilverCleanEntity'),
    (14, 'Sales.SalesTerritory','silver','fabmigration','fabmigration','bronze/adventureworks2','parquet','silver/adventureworks','01_SilverCleanEntity'),
    (15, 'Sales.SalesTerritoryHistory','silver','fabmigration','fabmigration','bronze/adventureworks2','parquet','silver/adventureworks','01_SilverCleanEntity'),
    (16, 'Sales.ShoppingCartItem','silver','fabmigration','fabmigration','bronze/adventureworks2','parquet','silver/adventureworks','01_SilverCleanEntity'),
    (17, 'Sales.SpecialOffer','silver','fabmigration','fabmigration','bronze/adventureworks2','parquet','silver/adventureworks','01_SilverCleanEntity'),
    (18, 'Sales.SpecialOfferProduct','silver','fabmigration','fabmigration','bronze/adventureworks2','parquet','silver/adventureworks','01_SilverCleanEntity'),
    (19, 'Sales.Store','silver','fabmigration','fabmigration','bronze/adventureworks2','parquet','silver/adventureworks','01_SilverCleanEntity'),

    (20,'HumanResources','silver',None,None,None,None,None,'nb_hr_bronze_to_silver'),
    (21,'Purchasing','silver',None,None,None,None,None,'nb_purchasing_bronze_to_silver'),
    (22,'Production1','silver',None,None,None,None,None,'nb_ProdTables'),
    (23,'Production2','silver',None,None,None,None,None,'nb_Production_med'),
    (24,'PersonAddress','silver',None,None,None,None,None,'silver_person_address'),
    (25,'PersonContactRoles','silver',None,None,None,None,None,'silver_person_contact_roles'),
    (26,'PersonEmail','silver',None,None,None,None,None,'silver_person_email'),
    (27,'PersonPhone','silver',None,None,None,None,None,'silver_person_phone')
]

columns = [
    "IDEntity","EntityName","Environment","ADLSaccount",
    "ContainerName","SourcePath","FileExtension",
    "TargetPath","Notebook"
]

entities_df = spark.createDataFrame(data, columns)

In [0]:
dbutils.widgets.text("Environment", "silver")
environment = dbutils.widgets.get("Environment")

# Equivalent to stored procedure filtering + CASE WHEN logic
metadata_df = (
    entities_df
    .filter(col("Environment") == environment)
    .select(
        col("ADLSaccount"),
        col("ContainerName").alias("containerName"),
        col("SourcePath").alias("sourceFolder"),
        col("TargetPath").alias("targetFolder"),
        col("Environment").alias("targetEnv"),
        col("EntityName").alias("entityName"),
        col("FileExtension").alias("fileExtension"),
        col("Notebook"),
        # Replicates: CASE WHEN FileExtension IS NULL THEN 0 ELSE 1 END
        when(col("FileExtension").isNull(), 0)
            .otherwise(1)
            .alias("NeedParameters")
    )
    .orderBy("entityName")
)

In [0]:
status = "Succeeded"

try:
    
    for row in metadata_df.toLocalIterator():
        
        item = row.asDict()
        notebook_path = item["Notebook"]
        need_parameters = bool(item["NeedParameters"])
        
        print(f"Running notebook: {notebook_path}")
        
        if need_parameters:
            
            params = {
                "ADLSaccount": item["ADLSaccount"],
                "containerName": item["containerName"],
                "sourceFolder": item["sourceFolder"],
                "targetFolder": item["targetFolder"],
                "targetEnv": item["targetEnv"],
                "entityName": item["entityName"],
                "fileExtension": item["fileExtension"]
            }
            
            dbutils.notebook.run(notebook_path, 43200, params)
        
        else:
            dbutils.notebook.run(notebook_path, 43200)

except Exception as e:
    status = "Failed"
    print("Pipeline failed:", str(e))

if status == "Failed":
    raise Exception("Execution failed")

In [0]:
end_time = datetime.now()

print(f"Pipeline finished with status: {status}")
print(f"End time: {end_time}")

if status == "Failed":
    raise Exception("Pipeline execution failed")

In [0]:
def log_end_execution(id_log: int, status: str):

    spark.sql(f"""
        MERGE INTO audit.tbl_log AS target
        USING (
            SELECT 
                {id_log} AS ID_Log,
                current_timestamp() AS EndTime,
                unix_timestamp(current_timestamp()) AS EndUnix,
                '{status}' AS Status
        ) AS source
        ON target.ID_Log = source.ID_Log
        
        WHEN MATCHED THEN UPDATE SET
            target.EndTime = source.EndTime,
            target.DurationSeconds = 
                source.EndUnix - unix_timestamp(target.StartTime),
            target.Status = source.Status
    """)